In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=1.0,
    max_retries=2,
)

In [4]:
# Create the state

class LLMState(TypedDict):
    
    question: str
    answer: str

In [5]:
def llm_qa(state: LLMState) -> LLMState:
    
    # Extract the question from the state
    question = state['question']
    
    # Form a prompt
    prompt = f'Answer the following question: {question}'
    
    # Ask the question to the model
    answer = model.invoke(prompt).content
    
    # Update the answer in the state
    state['answer'] = answer
    
    return state

In [6]:
# create the graph

graph = StateGraph(LLMState)

# Add Nodes
graph.add_node('llm_qa', llm_qa)

# Add Edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# Compile the graph
workflow = graph.compile()

In [7]:
# Execute the graph
initial_state = { 'question': 'What is the capital of India, and explain the reason behind it?' }

final_state = workflow.invoke(initial_state)

print(final_state)

{'question': 'What is the capital of India, and explain the reason behind it?', 'answer': [{'type': 'text', 'text': 'The capital of India is **New Delhi**. \n\nTo understand the reason why New Delhi is the capital, we have to look at the history of British rule in India, the strategic decisions made in the early 20th century, and the continuation of that choice after India gained independence.\n\nHere is the detailed explanation of the reasons behind choosing New Delhi as the capital:\n\n---\n\n### 1. The Shift from Calcutta (1911)\nUntil December 1911, **Calcutta** (now Kolkata) was the capital of British India. However, during the Delhi Durbar on December 12, 1911, King George V announced that the capital of the British Raj would be shifted from Calcutta to Delhi. \n\nThere were several major reasons for this shift:\n\n* **Geographical and Strategic Centrality:** Calcutta was located on the eastern edge of the subcontinent, making it difficult to govern the vast western, northern, an